# 第8周：Pseudobulk + edgeR 年龄组单模型

本 Notebook **只使用一个设计模型**：

```r
~ age_group + sex + technology
```

比较 **Old vs Young**，同时控制 sex 和 technology。

流程：
1. 读取 pseudobulk counts / metadata
2. 排除 Week 7 离群样本
3. 强制排除 `30-M-2_Lung_fibroblast/stromal`
4. 只保留 Young / Old
5. 使用 `edgeR::filterByExpr()`
6. TMM normalization
7. edgeR QL GLM
8. 检验 `age_groupold`
9. FDR < 0.05 且 |logFC| >= 1
10. 输出 DE 表、汇总表和火山图

**不进行连续年龄 `age_months` 模型。**

**火山图视觉格式严格按模板1：Down=青色、Normal=灰色、Up=橙红色，图例位于底部，Top 10 基因自动避让。**

**火山图输出路径已在 R 绘图前显式定义，避免 `找不到对象 volcano_file`。**


In [1]:
import shutil
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np

print("Rscript:", shutil.which("Rscript"))
if shutil.which("Rscript") is None:
    raise EnvironmentError("找不到 Rscript，请检查 R 是否已安装并加入 PATH。")

COUNT_FILE = Path("../data_processed/pseudobulk_counts.tsv")
META_FILE = Path("../data_processed/pseudobulk_metadata.tsv")
OUTLIER_FILE = Path("../results/pseudobulk/sample_outliers.csv")

OUT_DIR = Path("../results/de")
FIG_DIR = Path("../figures/de")
TMP_DIR = OUT_DIR / "tmp_edgeR"

for d in [OUT_DIR, FIG_DIR, TMP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

for f in [COUNT_FILE, META_FILE, OUTLIER_FILE]:
    print(f"{f}: {'✓ 存在' if f.exists() else '✗ 不存在'}")

if not all(f.exists() for f in [COUNT_FILE, META_FILE, OUTLIER_FILE]):
    raise FileNotFoundError("请检查上面的文件路径。")

Rscript: C:\Program Files\R\R-4.6.1\bin\x64\Rscript.EXE
..\data_processed\pseudobulk_counts.tsv: ✓ 存在
..\data_processed\pseudobulk_metadata.tsv: ✓ 存在
..\results\pseudobulk\sample_outliers.csv: ✓ 存在


In [2]:
counts = pd.read_csv(COUNT_FILE, sep="\t", index_col=0)
meta = pd.read_csv(META_FILE, sep="\t", index_col=0)
outlier_df = pd.read_csv(OUTLIER_FILE)

common = counts.columns.intersection(meta.index)
if len(common) == 0:
    raise ValueError("counts 和 metadata 没有共同 sample ID。")

counts = counts.loc[:, common]
meta = meta.loc[common].copy()

print("counts:", counts.shape)
print("metadata:", meta.shape)
print("metadata columns:", meta.columns.tolist())

counts: (3000, 110)
metadata: (110, 6)
metadata columns: ['mouse.id', 'tissue', 'major_cell_type', 'age_months', 'sex', 'cell_index']


## ④ 检查 metadata

必须存在：

- `tissue`
- `major_cell_type`
- `age_group`

如果没有 `age_group`，但存在 `age_months`，则自动生成：

- <= 3：young
- > 24：old
- 中间：middle

随后只保留 young 和 old。

In [3]:
required = ["tissue", "major_cell_type"]

missing = [x for x in required if x not in meta.columns]
if missing:
    raise KeyError("metadata 缺少必要字段: " + ", ".join(missing))

if "age_group" not in meta.columns:
    if "age_months" not in meta.columns:
        raise KeyError("metadata 同时缺少 age_group 和 age_months。")

    meta["age_months"] = pd.to_numeric(meta["age_months"], errors="coerce")

    if meta["age_months"].isna().any():
        raise ValueError("存在无法解析的 age_months。")

    # 冻结定义（config/analysis_parameters_v1.yaml）：young={1,3}, middle={18,21}, old={24,30}
    def _grp(a):
        if a in (1, 3):
            return "young"
        if a in (18, 21):
            return "middle"
        if a in (24, 30):
            return "old"
        return "other"
    meta["age_group"] = meta["age_months"].map(_grp)
else:
    meta["age_group"] = (meta["age_group"].astype(str).str.lower().str.strip())

print(meta["age_group"].value_counts(dropna=False))

print("sex:","存在" if "sex" in meta.columns else "不存在，将不加入模型")
print("technology:","存在" if "technology" in meta.columns else "不存在，将不加入模型")

age_group
middle    46
young     32
old       32
Name: count, dtype: int64
sex: 存在
technology: 不存在，将不加入模型


## ⑤ 排除离群样本

排除：

1. Week 7 `sample_outliers.csv` 中已经标记的样本
2. 强制排除 `30-M-2_Lung_fibroblast/stromal`

随后只保留 Young / Old。

In [4]:
outlier_ids = set(outlier_df.iloc[:, 0].dropna().astype(str))

matched_outliers = [
s for s in counts.columns
    if s in outlier_ids
]

forced = "30-M-2_Lung_fibroblast/stromal"

if forced in counts.columns and forced not in matched_outliers:
    matched_outliers.append(forced)

matched_outliers = list(dict.fromkeys(matched_outliers))

print("最终排除样本数:", len(matched_outliers))
for s in matched_outliers:
    print(" -", s)

excluded = meta.loc[[s for s in matched_outliers if s in meta.index]].copy()

if len(excluded):
    excluded.insert(0, "sample_id", excluded.index)
    excluded["outlier_reason"] = "Week 7 outlier / forced exclusion"

excluded.to_csv(OUT_DIR / "sample_exclusion_audit.tsv",sep="\t",index=False)

counts = counts.drop(columns=matched_outliers, errors="ignore")
meta = meta.drop(index=matched_outliers, errors="ignore")

keep_age = meta["age_group"].isin(["young", "old"])
counts = counts.loc[:, keep_age]
meta = meta.loc[keep_age].copy()

counts.to_csv(TMP_DIR / "counts.tsv", sep="\t")
meta.to_csv(TMP_DIR / "metadata.tsv", sep="\t")

print("\nYoung / Old 样本数:")
print(meta["age_group"].value_counts())

if forced in counts.columns:
    raise RuntimeError("30-M-2 仍然存在。")

最终排除样本数: 6
 - 1-M-62_Liver_fibroblast/stromal
 - 18-F-51_Liver_fibroblast/stromal
 - 21-F-54_Liver_fibroblast/stromal
 - 24-M-58_Liver_endothelial
 - 3-F-56_Liver_fibroblast/stromal
 - 30-M-2_Lung_fibroblast/stromal

Young / Old 样本数:
age_group
young    30
old      30
Name: count, dtype: int64


## ⑥ 生成 edgeR 单模型脚本

固定目标设计：

```r
~ age_group + sex + technology
```

目标系数：

```r
age_groupold
```

如果 sex / technology 不存在或只有一个水平，则自动省略。

如果设计矩阵不满秩，则优先尝试去掉 technology，再尝试去掉 sex。

In [5]:
def rpath(p):
    return str(p.resolve()).replace("\\", "/").replace('"', '\\"')

r_script = r"""
suppressPackageStartupMessages({
    library(edgeR)
    library(limma)
})

count_file <- "COUNT_FILE"
meta_file <- "META_FILE"
out_dir <- "OUT_DIR"
fig_dir <- "FIG_DIR"

dir.create(out_dir, recursive=TRUE, showWarnings=FALSE)
dir.create(fig_dir, recursive=TRUE, showWarnings=FALSE)

counts <- as.matrix(read.delim(count_file, row.names=1, check.names=FALSE, stringsAsFactors=FALSE))
storage.mode(counts) <- "integer"

meta <- read.delim(meta_file, row.names=1, check.names=FALSE, stringsAsFactors=FALSE)
meta <- meta[colnames(counts), , drop=FALSE]

standard_result <- function(
    tissue,
    cell_type,
    status="SKIP",
    reason=NA_character_,
    formula=NA_character_,
    n_samples=NA_integer_,
    n_young=NA_integer_,
    n_old=NA_integer_,
    residual_df=NA_integer_,
    genes_before=NA_integer_,
    genes_after_filterByExpr=NA_integer_,
    genes_removed=NA_integer_,
    significant_genes=NA_integer_,
    up_genes=NA_integer_,
    down_genes=NA_integer_
) {
    data.frame(
        tissue=as.character(tissue),
        cell_type=as.character(cell_type),
        status=as.character(status),
        reason=as.character(reason),
        formula=as.character(formula),
        n_samples=as.integer(n_samples),
        n_young=as.integer(n_young),
        n_old=as.integer(n_old),
        residual_df=as.integer(residual_df),
        genes_before=as.integer(genes_before),
        genes_after_filterByExpr=as.integer(genes_after_filterByExpr),
        genes_removed=as.integer(genes_removed),
        significant_genes=as.integer(significant_genes),
        up_genes=as.integer(up_genes),
        down_genes=as.integer(down_genes),
        stringsAsFactors=FALSE
    )
}

# ============================================================
# 单个 tissue × cell type
# ============================================================
run_one <- function(tissue_name, celltype_name) {
    keep <- (
        meta$tissue == tissue_name &
        meta$major_cell_type == celltype_name &
        meta$age_group %in% c("young", "old")
    )
    m <- meta[keep, , drop=FALSE]
    cts <- counts[, rownames(m), drop=FALSE]

    # ---------- 样本数检查 ----------
    if (nrow(m) < 4) {
        return(standard_result(tissue_name, celltype_name,
                               status="SKIP", reason="fewer than 4 samples",
                               n_samples=nrow(m)))
    }

    m$age_group <- factor(m$age_group, levels=c("young", "old"))
    ny <- sum(m$age_group == "young")
    no <- sum(m$age_group == "old")
    if (ny < 2 || no < 2) {
        return(standard_result(tissue_name, celltype_name,
                               status="SKIP", reason="fewer than 2 samples in young or old",
                               n_samples=nrow(m), n_young=ny, n_old=no))
    }

    # ---------- 设计矩阵 ----------
    vars <- "age_group"
    if ("sex" %in% colnames(m) && length(unique(na.omit(m$sex))) >= 2) {
        m$sex <- factor(m$sex)
        vars <- c(vars, "sex")
    }
    if ("technology" %in% colnames(m) && length(unique(na.omit(m$technology))) >= 2) {
        m$technology <- factor(m$technology)
        vars <- c(vars, "technology")
    }

    make_design <- function(v) {
        f <- as.formula(paste("~", paste(v, collapse=" + ")))
        model.matrix(f, data=m)
    }

    design <- make_design(vars)
    if (qr(design)$rank < ncol(design) && "technology" %in% vars) {
        vars <- setdiff(vars, "technology")
        design <- make_design(vars)
    }
    if (qr(design)$rank < ncol(design) && "sex" %in% vars) {
        vars <- setdiff(vars, "sex")
        design <- make_design(vars)
    }
    if (qr(design)$rank < ncol(design)) {
        return(standard_result(tissue_name, celltype_name,
                               status="SKIP", reason="design matrix not full rank",
                               n_samples=nrow(m), n_young=ny, n_old=no,
                               formula=paste("~", paste(vars, collapse=" + "))))
    }

    residual_df <- nrow(design) - qr(design)$rank
    if (residual_df < 1) {
        return(standard_result(tissue_name, celltype_name,
                               status="SKIP", reason="insufficient residual df",
                               n_samples=nrow(m), n_young=ny, n_old=no,
                               residual_df=residual_df))
    }

    # ---------- edgeR 核心 ----------
    y <- DGEList(counts=cts)
    keep_gene <- filterByExpr(y, group=m$age_group)
    y <- y[keep_gene, , keep.lib.sizes=FALSE]
    if (nrow(y) < 10) {
        return(standard_result(
            tissue_name, 
            celltype_name,
            status="SKIP", 
            reason="fewer than 10 genes after filterByExpr",
            n_samples=nrow(m), 
            n_young=ny, 
            n_old=no,
            residual_df=residual_df,
            genes_before=nrow(cts),
            genes_after_filterByExpr=nrow(y),
            genes_removed=nrow(cts)-nrow(y))
        )
    }

    y <- normLibSizes(y, method="TMM")
    y <- estimateDisp(y, design)
    fit <- glmQLFit(y, design, robust=TRUE)

    coef_id <- which(colnames(design) == "age_groupold")
    if (length(coef_id) != 1) {
        return(standard_result(
                    tissue_name, 
                    celltype_name,
                    status="SKIP", 
                    reason="age_groupold coefficient unavailable")
                )
    }

    qlf <- glmQLFTest(fit, coef=coef_id)
    tab <- topTags(qlf, n=Inf, sort.by="PValue")$table
    tab$gene <- rownames(tab)
    tab$tissue <- tissue_name
    tab$cell_type <- celltype_name

    # ---------- 显著性定义 ----------
    tab$significant <- (tab$FDR < 0.05)
    tab$direction <- "Normal"
    tab$direction[tab$significant & tab$logFC > 0] <- "Up"
    tab$direction[tab$significant & tab$logFC < 0] <- "Down"

    # ---------- 保存 DE 表格 ----------
    st <- gsub("[^A-Za-z0-9_-]", "_", tissue_name)
    ct <- gsub("[^A-Za-z0-9_-]", "_", celltype_name)
    de_file <- file.path(out_dir, paste0(st, "_", ct, "_old_vs_young_edgeR.tsv"))
    write.table(tab, de_file, sep="\t", quote=FALSE, row.names=FALSE)

    # ---------- 火山图：统一使用自定义 ggplot2 + ggrepel ----------
    # 准备数据
    volcano_data <- tab[is.finite(tab$logFC) & is.finite(tab$FDR) & !is.na(tab$gene), , drop=FALSE]
    if (nrow(volcano_data) == 0) {
        # 没有有效数据，跳过绘图，但标记为 OK（因为 DE 分析已完成）
        return(standard_result(
            tissue_name, celltype_name,
            status = "OK (volcano skipped)",
            reason = "No finite logFC/FDR",
            formula = paste("~", paste(vars, collapse = " + ")),
            n_samples = nrow(m),
            n_young = ny,
            n_old = no,
            residual_df = residual_df,
            genes_before = nrow(cts),
            genes_after_filterByExpr = nrow(y),
            genes_removed = nrow(cts) - nrow(y),
            significant_genes = sum(tab$significant),
            up_genes = sum(tab$direction == "Up", na.rm = TRUE),
            down_genes = sum(tab$direction == "Down", na.rm = TRUE)
        ))
    }

    volcano_data$FDR <- pmax(volcano_data$FDR, .Machine$double.xmin)
    volcano_data$logFC <- as.numeric(volcano_data$logFC)
    volcano_data$FDR <- as.numeric(volcano_data$FDR)
    volcano_data$gene <- as.character(volcano_data$gene)

    # 分类
    volcano_data$regulate <- "Normal"
    volcano_data$regulate[volcano_data$FDR < 0.05 & volcano_data$logFC < 0] <- "Down"
    volcano_data$regulate[volcano_data$FDR < 0.05 & volcano_data$logFC > 0] <- "Up"
    volcano_data$regulate <- factor(volcano_data$regulate, levels = c("Down", "Normal", "Up"))

    # 选出 Top 10 最显著基因（按FDR排序，上下各5个优先）
    sig <- volcano_data[volcano_data$FDR < 0.05, , drop=FALSE]
    label_tab <- data.frame()
    if (nrow(sig) > 0) {
        sig_up <- sig[sig$logFC > 0, , drop=FALSE]
        sig_down <- sig[sig$logFC < 0, , drop=FALSE]
        sig_up <- sig_up[order(sig_up$FDR, -abs(sig_up$logFC)), , drop=FALSE]
        sig_down <- sig_down[order(sig_down$FDR, -abs(sig_down$logFC)), , drop=FALSE]
        label_tab <- rbind(
            head(sig_down, 5),
            head(sig_up, 5)
        )
        if (nrow(label_tab) < min(10, nrow(sig))) {
            remaining <- sig[!(sig$gene %in% label_tab$gene), , drop=FALSE]
            remaining <- remaining[order(remaining$FDR, -abs(remaining$logFC)), , drop=FALSE]
            need <- min(10, nrow(sig)) - nrow(label_tab)
            if (need > 0 && nrow(remaining) > 0) {
                label_tab <- rbind(label_tab, head(remaining, need))
            }
        }
        label_tab <- label_tab[!duplicated(label_tab$gene), , drop=FALSE]
    }

    # 绘制基础图（图例顶部水平）
    p1 <- ggplot2::ggplot(volcano_data, ggplot2::aes(x = logFC, y = -log10(FDR), fill = regulate)) +
        ggplot2::geom_point(shape = 21, size = 1.5, color = "white", stroke = 0.3, alpha = 0.8) +
        ggplot2::scale_fill_manual(
            values = c(Down = "#00AFBB", Normal = "#999999", Up = "#FC4E07"),
            limits = c("Down", "Normal", "Up"),
            drop = FALSE
        ) +
        ggplot2::geom_vline(xintercept = c(-1, 1), linetype = "dashed", linewidth = 0.5, color = "black") +
        ggplot2::geom_hline(yintercept = -log10(0.05), linetype = "dashed", linewidth = 0.5, color = "black") +
        ggplot2::labs(
            title = paste("Old vs Young:", tissue_name, "|", celltype_name),
            x = expression(Log[2]*FC),
            y = expression(-Log[10]*FDR),
            fill = "Regulate"
        ) +
        ggplot2::theme_bw(base_size = 12) +
        ggplot2::theme(
            plot.title = ggplot2::element_text(hjust = 0.5, face = "bold", size = 14),
            legend.position = "top",
            legend.justification = "left",
            legend.direction = "horizontal",
            legend.box = "horizontal",
            legend.title = ggplot2::element_text(size = 12),
            legend.text = ggplot2::element_text(size = 11),
            legend.key.width = ggplot2::unit(1.0, "cm"),
            legend.background = ggplot2::element_rect(fill = "white", color = NA),
            legend.margin = ggplot2::margin(t = 2, r = 0, b = 0, l = 0),
            panel.grid.major = ggplot2::element_line(color = "#E5E5E5", linewidth = 0.6),
            panel.grid.minor = ggplot2::element_line(color = "#F0F0F0", linewidth = 0.35),
            panel.border = ggplot2::element_rect(color = "#666666", fill = NA, linewidth = 0.7),
            axis.title = ggplot2::element_text(size = 16, color = "black"),
            axis.text = ggplot2::element_text(size = 11, color = "black"),
            plot.margin = ggplot2::margin(20, 12, 8, 12)   # 顶部留空给图例
        ) +
        ggplot2::guides(fill = ggplot2::guide_legend(
            override.aes = list(shape = 21, size = 4, color = "white")
        ))

    # 添加标签（使用 ggrepel 自动避让）
    if (nrow(label_tab) > 0) {
        p1 <- p1 + ggrepel::geom_text_repel(
            data = label_tab,
            ggplot2::aes(x = logFC, y = -log10(FDR), label = gene),
            inherit.aes = FALSE,
            size = 3.5,
            color = "black",
            box.padding = 0.5,
            point.padding = 0.3,
            force = 2,
            force_pull = 0.5,
            max.overlaps = Inf,
            segment.color = "grey50",
            segment.size = 0.3,
            seed = 123
        )
    }

    # 调整坐标范围，确保所有点可见
    x_rng <- range(volcano_data$logFC, na.rm = TRUE)
    y_rng <- range(-log10(volcano_data$FDR), na.rm = TRUE)
    x_pad <- max(0.5, diff(x_rng) * 0.08)
    y_pad <- max(0.5, diff(y_rng) * 0.08)
    p1 <- p1 + ggplot2::coord_cartesian(
        xlim = c(x_rng[1] - x_pad, x_rng[2] + x_pad),
        ylim = c(0, y_rng[2] + y_pad),
        clip = "off"
    )

    # 保存 PDF
    volcano_file <- file.path(fig_dir, paste0(st, "_", ct, "_old_vs_young_volcano.pdf"))
    ggplot2::ggsave(filename = volcano_file, plot = p1,
                    dpi = 300, width = 6, height = 5,
                    units = "in", device = "pdf")

    # 检查是否成功生成
    if (!file.exists(volcano_file) || file.info(volcano_file)$size <= 0) {
        return(standard_result(
            tissue_name, celltype_name,
            status = "ERROR",
            reason = paste("PDF not created:", volcano_file),
            formula = paste("~", paste(vars, collapse = " + ")),
            n_samples = nrow(m),
            n_young = ny,
            n_old = no,
            residual_df = residual_df,
            genes_before = nrow(cts),
            genes_after_filterByExpr = nrow(y),
            genes_removed = nrow(cts) - nrow(y),
            significant_genes = sum(tab$significant),
            up_genes = sum(tab$direction == "Up", na.rm = TRUE),
            down_genes = sum(tab$direction == "Down", na.rm = TRUE)
        ))
    }

    # ---------- 汇总 ----------
    standard_result(
        tissue_name, celltype_name,
        status = "OK",
        reason = NA_character_,
        formula = paste("~", paste(vars, collapse = " + ")),
        n_samples = nrow(m),
        n_young = ny,
        n_old = no,
        residual_df = residual_df,
        genes_before = nrow(cts),
        genes_after_filterByExpr = nrow(y),
        genes_removed = nrow(cts) - nrow(y),
        significant_genes = sum(tab$significant),
        up_genes = sum(tab$direction == "Up", na.rm = TRUE),
        down_genes = sum(tab$direction == "Down", na.rm = TRUE)
    )
}

# ============================================================
# 批量运行
# ============================================================
results <- list()
k <- 1
for (t in unique(meta$tissue)) {
    for (ct in unique(meta$major_cell_type)) {
        results[[k]] <- tryCatch(
            run_one(t, ct),
            error = function(e) {
                standard_result(tissue = t, cell_type = ct,
                                status = "ERROR", reason = e$message)
            }
        )
        k <- k + 1
    }
}

summary_df <- do.call(rbind, results)
write.table(summary_df, file.path(out_dir, "DE_summary_old_vs_young.tsv"),
            sep = "\t", quote = FALSE, row.names = FALSE)
print(summary_df)
cat("\nOld vs Young edgeR analysis completed.\n")
"""

# 替换路径
r_script = r_script.replace("COUNT_FILE", rpath(TMP_DIR / "counts.tsv"))
r_script = r_script.replace("META_FILE", rpath(TMP_DIR / "metadata.tsv"))
r_script = r_script.replace("OUT_DIR", rpath(OUT_DIR))
r_script = r_script.replace("FIG_DIR", rpath(FIG_DIR))

r_file = TMP_DIR / "run_edgeR_old_vs_young.R"
r_file.write_text(r_script, encoding="utf-8")

print("R 脚本已生成:", r_file)

R 脚本已生成: ..\results\de\tmp_edgeR\run_edgeR_old_vs_young.R


## ⑦ 运行 edgeR

固定检验：

```r
~ age_group + sex + technology
```

目标系数：

```r
age_groupold
```

因此：

- `logFC > 0`：Old 高于 Young
- `logFC < 0`：Young 高于 Old

> **修正版说明：** 不同组织/细胞类型如果因为样本数、设计矩阵或自由度不足而 `SKIP`，现在会返回完全相同的汇总字段，因此不会再出现 `rbind` “变量的列数不正确”。同时新版 edgeR 使用 `normLibSizes(..., method="TMM")`。


In [ ]:
# 正式运行 edgeR + 火山图
res = subprocess.run(
    ["Rscript", str(r_file)],
    capture_output=True,
    text=True
)

print("=== edgeR / 火山图运行输出 ===")
print(res.stdout)

if res.returncode != 0:
    print("\n--- R ERROR (完整 stderr) ---")
    print(res.stderr)
    err_file = TMP_DIR / "edgeR_last_error.txt"
    err_file.write_text(res.stderr or "<stderr为空>", encoding="utf-8")
    print("R 错误已保存：", err_file.resolve())
    raise RuntimeError("edgeR 分析或火山图生成失败；请查看上面的 R ERROR。")

# 强制检查 PDF 是否真实生成
pdfs = sorted(FIG_DIR.glob("*old_vs_young_volcano.pdf"))
print("\n=== 实际生成的火山图 ===")
print("目录：", FIG_DIR.resolve())
print("数量：", len(pdfs))

for f in pdfs:
    print(f"✓ {f.resolve()}  |  {f.stat().st_size / 1024:.1f} KB")

if len(pdfs) == 0:
    print("\n--- 当前目录没有生成任何火山图 PDF ---")
    if res.stderr:
        print(res.stderr)
    raise FileNotFoundError(f"没有生成火山图 PDF：{FIG_DIR.resolve()}")

print("\n✓ Old vs Young edgeR + 火山图分析完成。")


=== edgeR / 火山图运行输出 ===
            tissue          cell_type status               reason
1           Kidney        endothelial     OK                 <NA>
2           Kidney fibroblast/stromal     OK                 <NA>
3            Liver        endothelial     OK                 <NA>
4            Liver fibroblast/stromal   SKIP fewer than 4 samples
5             Lung        endothelial     OK                 <NA>
6             Lung fibroblast/stromal     OK                 <NA>
7  Heart_and_Aorta        endothelial     OK                 <NA>
8  Heart_and_Aorta fibroblast/stromal     OK                 <NA>
9              Fat        endothelial   SKIP fewer than 4 samples
10             Fat fibroblast/stromal   SKIP fewer than 4 samples
             formula n_samples n_young n_old residual_df genes_before
1  ~ age_group + sex         9       5     4           6         3000
2  ~ age_group + sex         9       5     4           6         3000
3  ~ age_group + sex         9       5  

In [7]:
summary_file = OUT_DIR / "DE_summary_old_vs_young.tsv"

if not summary_file.exists():
    raise FileNotFoundError("没有找到 DE_summary_old_vs_young.tsv")

summary_df = pd.read_csv(summary_file,sep="\t")

display(summary_df)

,tissue,cell_type,status,reason,formula,n_samples,n_young,n_old,residual_df,genes_before,genes_after_filterByExpr,genes_removed,significant_genes,up_genes,down_genes
0,Kidney,endothelial,OK,NaN,~ age_group + sex,9,5.0,4.0,6.0,3000.0,1158.0,1842.0,80.0,27.0,53.0
1,Kidney,fibroblast/stromal,OK,NaN,~ age_group + sex,9,5.0,4.0,6.0,3000.0,906.0,2094.0,341.0,141.0,200.0
2,Liver,endothelial,OK,NaN,~ age_group + sex,9,5.0,4.0,6.0,3000.0,399.0,2601.0,19.0,7.0,12.0
3,Liver,fibroblast/stromal,SKIP,fewer than 4 samples,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Lung,endothelial,OK,NaN,~ age_group + sex,8,4.0,4.0,5.0,3000.0,576.0,2424.0,25.0,18.0,7.0
5,Lung,fibroblast/stromal,OK,NaN,~ age_group + sex,9,6.0,3.0,6.0,3000.0,1022.0,1978.0,42.0,30.0,12.0
6,Heart_and_Aorta,endothelial,OK,NaN,~ age_group + sex,6,2.0,4.0,3.0,3000.0,1315.0,1685.0,1.0,0.0,1.0
7,Heart_and_Aorta,fibroblast/stromal,OK,NaN,~ age_group + sex,6,2.0,4.0,3.0,3000.0,1398.0,1602.0,9.0,2.0,7.0
8,Fat,endothelial,SKIP,fewer than 4 samples,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Fat,fibroblast/stromal,SKIP,fewer than 4 samples,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
summary_file = OUT_DIR / "DE_summary_old_vs_young.tsv"

if not summary_file.exists():
    raise FileNotFoundError(f"R 没有生成结果文件：{summary_file}\n""请先检查前一个 edgeR Cell 的 R 输出。")

summary_df = pd.read_csv(summary_file,sep="\t")

print("结果文件：", summary_file)
print("结果维度：", summary_df.shape)
print("结果列：")
print(summary_df.columns.tolist())

display(summary_df)

if "status" in summary_df.columns:
    print("\n=== 正常完成 ===")
    display(summary_df[summary_df["status"] == "OK"])

    print("\n=== SKIP / ERROR ===")
    display(summary_df[summary_df["status"] != "OK"])
else:
    print("\n⚠️ 结果文件中没有 status 列。")
    print("说明 R 端输出格式仍然不符合预期，请检查上一个 Cell 的 R 输出。")

结果文件： ..\results\de\DE_summary_old_vs_young.tsv
结果维度： (10, 15)
结果列：
['tissue', 'cell_type', 'status', 'reason', 'formula', 'n_samples', 'n_young', 'n_old', 'residual_df', 'genes_before', 'genes_after_filterByExpr', 'genes_removed', 'significant_genes', 'up_genes', 'down_genes']


,tissue,cell_type,status,reason,formula,n_samples,n_young,n_old,residual_df,genes_before,genes_after_filterByExpr,genes_removed,significant_genes,up_genes,down_genes
0,Kidney,endothelial,OK,NaN,~ age_group + sex,9,5.0,4.0,6.0,3000.0,1158.0,1842.0,80.0,27.0,53.0
1,Kidney,fibroblast/stromal,OK,NaN,~ age_group + sex,9,5.0,4.0,6.0,3000.0,906.0,2094.0,341.0,141.0,200.0
2,Liver,endothelial,OK,NaN,~ age_group + sex,9,5.0,4.0,6.0,3000.0,399.0,2601.0,19.0,7.0,12.0
3,Liver,fibroblast/stromal,SKIP,fewer than 4 samples,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Lung,endothelial,OK,NaN,~ age_group + sex,8,4.0,4.0,5.0,3000.0,576.0,2424.0,25.0,18.0,7.0
5,Lung,fibroblast/stromal,OK,NaN,~ age_group + sex,9,6.0,3.0,6.0,3000.0,1022.0,1978.0,42.0,30.0,12.0
6,Heart_and_Aorta,endothelial,OK,NaN,~ age_group + sex,6,2.0,4.0,3.0,3000.0,1315.0,1685.0,1.0,0.0,1.0
7,Heart_and_Aorta,fibroblast/stromal,OK,NaN,~ age_group + sex,6,2.0,4.0,3.0,3000.0,1398.0,1602.0,9.0,2.0,7.0
8,Fat,endothelial,SKIP,fewer than 4 samples,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Fat,fibroblast/stromal,SKIP,fewer than 4 samples,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



=== 正常完成 ===


,tissue,cell_type,status,reason,formula,n_samples,n_young,n_old,residual_df,genes_before,genes_after_filterByExpr,genes_removed,significant_genes,up_genes,down_genes
0,Kidney,endothelial,OK,NaN,~ age_group + sex,9,5.0,4.0,6.0,3000.0,1158.0,1842.0,80.0,27.0,53.0
1,Kidney,fibroblast/stromal,OK,NaN,~ age_group + sex,9,5.0,4.0,6.0,3000.0,906.0,2094.0,341.0,141.0,200.0
2,Liver,endothelial,OK,NaN,~ age_group + sex,9,5.0,4.0,6.0,3000.0,399.0,2601.0,19.0,7.0,12.0
4,Lung,endothelial,OK,NaN,~ age_group + sex,8,4.0,4.0,5.0,3000.0,576.0,2424.0,25.0,18.0,7.0
5,Lung,fibroblast/stromal,OK,NaN,~ age_group + sex,9,6.0,3.0,6.0,3000.0,1022.0,1978.0,42.0,30.0,12.0
6,Heart_and_Aorta,endothelial,OK,NaN,~ age_group + sex,6,2.0,4.0,3.0,3000.0,1315.0,1685.0,1.0,0.0,1.0
7,Heart_and_Aorta,fibroblast/stromal,OK,NaN,~ age_group + sex,6,2.0,4.0,3.0,3000.0,1398.0,1602.0,9.0,2.0,7.0



=== SKIP / ERROR ===


,tissue,cell_type,status,reason,formula,n_samples,n_young,n_old,residual_df,genes_before,genes_after_filterByExpr,genes_removed,significant_genes,up_genes,down_genes
3,Liver,fibroblast/stromal,SKIP,fewer than 4 samples,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Fat,endothelial,SKIP,fewer than 4 samples,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Fat,fibroblast/stromal,SKIP,fewer than 4 samples,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 最终分析标准

本周只采用：

```text
~ age_group + sex + technology
```

显著差异基因：

```text
FDR < 0.05
且
|logFC| >= 1
```

其中 `age_groupold` 是主要生物学检验。

**不会再生成连续年龄 `age_months` 的分析结果。**
